# 🏷️ Phase 3: Brand Logo Detection

**Samsung PRISM - Harmful Content Detection Pipeline**

This notebook trains a YOLOv8 model for detecting competitor brand logos.

## Target Logos (Samsung Competitors)
- 🍎 Apple
- 🔍 Google
- 📱 Huawei
- 📱 Xiaomi
- 📱 OnePlus
- 📱 Oppo
- 📱 Vivo
- 📺 Sony
- 📺 LG

---

**⚠️ IMPORTANT: Enable GPU in Colab**
- Go to `Runtime` → `Change runtime type` → Select `GPU`

## 1️⃣ Setup & Installation

In [ ]:
# Install required packages
!pip install ultralytics>=8.0.200 -q
!pip install roboflow -q

# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import libraries
import os
import yaml
import shutil
import random
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display
import matplotlib.pyplot as plt
import numpy as np
import cv2
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

## 2️⃣ Dataset Download Instructions

### Option A: Manual Download from Kaggle

| Dataset | Link | Description |
|---------|------|-------------|
| LogoDet-3K | [kaggle.com/lyly99/logodet3k](https://www.kaggle.com/datasets/lyly99/logodet3k) | 3000+ logo classes |
| FlickrLogos-32 | [kaggle.com/kozodoi/flickrlogos32](https://www.kaggle.com/datasets/kozodoi/flickrlogos32) | 32 logo classes |

### Option B: Use Roboflow (Recommended)

Search Roboflow Universe for logo detection datasets with your target brands.

In [ ]:
# Create data directories
!mkdir -p /content/data/logos/train/images
!mkdir -p /content/data/logos/train/labels
!mkdir -p /content/data/logos/valid/images
!mkdir -p /content/data/logos/valid/labels
!mkdir -p /content/data/logos/test/images
!mkdir -p /content/data/logos/test/labels
!mkdir -p /content/runs

print("✓ Directory structure created")

In [ ]:
# Option: Download from Roboflow
from roboflow import Roboflow

# Initialize Roboflow
# Get your API key from https://roboflow.com/
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")

# Example: Download a logo detection dataset
# project = rf.workspace("your-workspace").project("logo-detection")
# dataset = project.version(1).download("yolov8")

print("⚠️ Replace with your Roboflow API key to download datasets")

In [ ]:
# ALTERNATIVE: Manual dataset upload
# Upload your logo dataset zip file and extract

# from google.colab import files
# uploaded = files.upload()  # Upload your dataset zip

# Extract uploaded file
# import zipfile
# with zipfile.ZipFile('logo_dataset.zip', 'r') as zip_ref:
#     zip_ref.extractall('/content/data/logos/')

print("Manual upload instructions:")
print("1. Download logo datasets from Kaggle")
print("2. Filter for target competitor brands")
print("3. Convert annotations to YOLO format")
print("4. Upload to Colab or Google Drive")

## 3️⃣ Dataset Configuration

In [ ]:
# Define target logo classes (Samsung competitors)
LOGO_CLASSES = {
    0: 'apple',
    1: 'google',
    2: 'huawei',
    3: 'xiaomi',
    4: 'oneplus',
    5: 'oppo',
    6: 'vivo',
    7: 'sony',
    8: 'lg',
    9: 'samsung'  # Own brand for reference
}

# Brands to flag as COMPETITOR (UNSAFE)
COMPETITOR_BRANDS = ['apple', 'google', 'huawei', 'xiaomi', 
                      'oneplus', 'oppo', 'vivo', 'sony', 'lg']

print("Target Logo Classes:")
for i, name in LOGO_CLASSES.items():
    competitor = "⚠️ COMPETITOR" if name in COMPETITOR_BRANDS else "✓ Own brand"
    print(f"  {i}: {name:<10} {competitor}")

In [ ]:
# Create data.yaml for YOLOv8 training
data_yaml = """
# Logo Detection Dataset Configuration
path: /content/data/logos
train: train/images
val: valid/images
test: test/images

# Classes - Samsung Competitors
names:
  0: apple
  1: google
  2: huawei
  3: xiaomi
  4: oneplus
  5: oppo
  6: vivo
  7: sony
  8: lg
  9: samsung

# Number of classes
nc: 10
"""

# Save configuration
with open('/content/data/logos/data.yaml', 'w') as f:
    f.write(data_yaml)

print("✓ Created data.yaml configuration")
print(data_yaml)

In [ ]:
# Helper: Create synthetic logo images for demo/testing
# (Replace with real data for production training)

from PIL import Image, ImageDraw, ImageFont

def create_sample_logo_image(brand_name, output_path):
    """Create a simple sample logo image for testing."""
    # Create image
    img = Image.new('RGB', (400, 400), color='white')
    draw = ImageDraw.Draw(img)
    
    # Add brand text as placeholder
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 50)
    except:
        font = ImageFont.load_default()
    
    # Draw brand name
    draw.rectangle([50, 100, 350, 300], outline='black', width=3)
    draw.text((100, 180), brand_name.upper(), fill='black', font=font)
    
    # Save image
    img.save(output_path)
    return output_path

# Create sample images for each brand (for demo purposes)
print("Creating sample logo images for demo...")
for brand in LOGO_CLASSES.values():
    output_path = f"/content/data/logos/train/images/{brand}_sample.jpg"
    create_sample_logo_image(brand, output_path)
    
    # Create corresponding YOLO label (full image bounding box)
    label_path = f"/content/data/logos/train/labels/{brand}_sample.txt"
    class_id = list(LOGO_CLASSES.values()).index(brand)
    with open(label_path, 'w') as f:
        # YOLO format: class x_center y_center width height (normalized)
        f.write(f"{class_id} 0.5 0.5 0.75 0.5\n")

print(f"✓ Created {len(LOGO_CLASSES)} sample logo images")

## 4️⃣ Data Visualization

In [ ]:
# Visualize sample logo images
train_images_dir = '/content/data/logos/train/images'
train_labels_dir = '/content/data/logos/train/labels'

if os.path.exists(train_images_dir):
    images = [f for f in os.listdir(train_images_dir) if f.endswith(('.jpg', '.png'))]
    samples = images[:min(9, len(images))]
    
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    
    for ax, img_name in zip(axes.flat, samples):
        img_path = os.path.join(train_images_dir, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        ax.imshow(img)
        ax.set_title(img_name.replace('_sample.jpg', '').upper())
        ax.axis('off')
    
    # Hide empty subplots
    for i in range(len(samples), 9):
        axes.flat[i].axis('off')
    
    plt.suptitle('Sample Logo Images', fontsize=14)
    plt.tight_layout()
    plt.savefig('/content/sample_logos.png', dpi=150)
    plt.show()

## 5️⃣ Model Training

In [ ]:
# Load YOLOv8 pretrained model
MODEL_SIZE = 'yolov8m'  # Medium model

model = YOLO(f'{MODEL_SIZE}.pt')
print(f"✓ Loaded {MODEL_SIZE} pretrained weights")

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'data': '/content/data/logos/data.yaml',
    'epochs': 50,                # Training epochs
    'imgsz': 640,                # Image size
    'batch': 16,                 # Batch size
    'device': 0,                 # GPU device
    'workers': 4,                # Data loader workers
    'patience': 10,              # Early stopping patience
    'save': True,                # Save checkpoints
    'project': '/content/runs',  # Output directory
    'name': 'logo_detection',    # Run name
    'exist_ok': True,            # Overwrite existing
    'pretrained': True,          # Use pretrained weights
    'optimizer': 'AdamW',        # Optimizer
    'lr0': 0.01,                 # Initial learning rate
    'mosaic': 1.0,               # Mosaic augmentation
    'mixup': 0.1,                # Mixup augmentation
    'fliplr': 0.5,               # Flip left-right
}

print("Training Configuration:")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# 🚀 Start Training
print("\n" + "="*50)
print("🚀 STARTING LOGO DETECTION MODEL TRAINING")
print("="*50 + "\n")

# Note: For production, use real logo dataset with proper annotations
# This demo uses synthetic images

results = model.train(**TRAINING_CONFIG)

print("\n" + "="*50)
print("✓ TRAINING COMPLETED")
print("="*50)

## 6️⃣ Model Evaluation

In [ ]:
# Load best model
best_model_path = '/content/runs/logo_detection/weights/best.pt'
trained_model = YOLO(best_model_path)

print(f"✓ Loaded best model from: {best_model_path}")

In [ ]:
# Validate model
metrics = trained_model.val(
    data='/content/data/logos/data.yaml',
    batch=16,
    imgsz=640,
    device=0
)

print("\n" + "="*50)
print("📊 EVALUATION METRICS")
print("="*50)
print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")
print("="*50)

In [ ]:
# Display training results
results_dir = '/content/runs/logo_detection'

if os.path.exists(f'{results_dir}/results.png'):
    display(Image(filename=f'{results_dir}/results.png', width=800))

if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    display(Image(filename=f'{results_dir}/confusion_matrix.png', width=600))

## 7️⃣ Inference Pipeline

In [ ]:
# Logo detection function
def detect_logos(model, image_path, confidence_threshold=0.6):
    """
    Detect logos in an image and flag competitors.
    
    Args:
        model: Trained YOLOv8 model
        image_path: Path to image
        confidence_threshold: Minimum confidence
        
    Returns:
        Detection result with competitor flags
    """
    # Run inference
    results = model(image_path, verbose=False)[0]
    
    detections = []
    competitor_detections = []
    samsung_detected = False
    
    for box in results.boxes:
        conf = float(box.conf[0])
        if conf >= confidence_threshold:
            cls_id = int(box.cls[0])
            brand = results.names.get(cls_id, f"class_{cls_id}").lower()
            bbox = box.xyxy[0].cpu().numpy().tolist()
            
            detection = {
                'brand': brand,
                'class_id': cls_id,
                'confidence': round(conf, 4),
                'is_competitor': brand in COMPETITOR_BRANDS,
                'bbox': {
                    'x1': int(bbox[0]),
                    'y1': int(bbox[1]),
                    'x2': int(bbox[2]),
                    'y2': int(bbox[3])
                }
            }
            
            detections.append(detection)
            
            if brand in COMPETITOR_BRANDS:
                competitor_detections.append(detection)
            elif brand == 'samsung':
                samsung_detected = True
    
    return {
        'detected': len(detections) > 0,
        'competitor_detected': len(competitor_detections) > 0,
        'samsung_detected': samsung_detected,
        'detections': detections,
        'competitor_detections': competitor_detections,
        'decision': 'UNSAFE' if competitor_detections else 'SAFE',
        'visualization': results.plot()
    }

print("✓ Logo detection function defined")

In [ ]:
# Test logo detection
test_image = '/content/data/logos/train/images/apple_sample.jpg'

if os.path.exists(test_image):
    result = detect_logos(trained_model, test_image)
    
    print("\n" + "="*50)
    print("🏷️ LOGO DETECTION RESULT")
    print("="*50)
    print(f"Logos detected: {result['detected']}")
    print(f"Competitor detected: {result['competitor_detected']}")
    print(f"Samsung detected: {result['samsung_detected']}")
    
    if result['detections']:
        print("\nDetections:")
        for d in result['detections']:
            status = "⚠️ COMPETITOR" if d['is_competitor'] else "✓"
            print(f"  {status} {d['brand']} ({d['confidence']:.2%})")
    
    print(f"\n🚨 FINAL DECISION: {result['decision']}")
    
    # Display
    vis_img = cv2.cvtColor(result['visualization'], cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 8))
    plt.imshow(vis_img)
    plt.title(f"Decision: {result['decision']}")
    plt.axis('off')
    plt.show()

In [ ]:
# Test on multiple logo images
print("\n" + "="*50)
print("🧪 TESTING LOGO DETECTION")
print("="*50 + "\n")

test_images_dir = '/content/data/logos/train/images'
if os.path.exists(test_images_dir):
    test_images = os.listdir(test_images_dir)
    
    for img_name in test_images[:5]:
        img_path = os.path.join(test_images_dir, img_name)
        result = detect_logos(trained_model, img_path)
        
        brand = img_name.replace('_sample.jpg', '')
        status = "🔴 UNSAFE" if result['decision'] == 'UNSAFE' else "🟢 SAFE"
        
        if result['competitor_detected']:
            competitors = [d['brand'] for d in result['competitor_detections']]
            print(f"{status} | {brand:<10} | Competitors: {', '.join(competitors)}")
        else:
            print(f"{status} | {brand:<10} | No competitor logos")

## 8️⃣ Save Model

In [ ]:
# Save best model to Google Drive
drive_path = '/content/drive/MyDrive/samsung_prism/models/logo_detector'
os.makedirs(drive_path, exist_ok=True)

# Copy weights
shutil.copy(best_model_path, f'{drive_path}/best.pt')

# Save results
if os.path.exists('/content/runs/logo_detection/results.csv'):
    shutil.copy('/content/runs/logo_detection/results.csv', f'{drive_path}/results.csv')

print(f"\n✓ Model saved to: {drive_path}")
print("\nSaved files:")
for f in os.listdir(drive_path):
    size_mb = os.path.getsize(f'{drive_path}/{f}') / (1024*1024)
    print(f"  - {f} ({size_mb:.2f} MB)")

In [ ]:
# Example output format
import json

sample_output = {
    "status": "UNSAFE",
    "detection_type": "logo",
    "competitor_detected": True,
    "detections": [
        {
            "brand": "apple",
            "confidence": 0.92,
            "is_competitor": True,
            "bbox": {"x1": 100, "y1": 150, "x2": 200, "y2": 250}
        }
    ],
    "explanation": "Competitor logo detected: Apple (92% confidence)"
}

print("\n📋 Sample Output Format:")
print(json.dumps(sample_output, indent=2))

In [ ]:
print("\n✓ Logo Detection Model Training Complete!")
print("\nNext Steps:")
print("  Run 04_unified_pipeline.ipynb for end-to-end inference")

---

## Summary

| Metric | Value |
|--------|-------|
| Model | YOLOv8m |
| Classes | 10 (9 competitors + Samsung) |
| mAP@50 | [See above] |
| Competitors | Apple, Google, Huawei, Xiaomi, OnePlus, Oppo, Vivo, Sony, LG |

**Model saved to:** `/content/drive/MyDrive/samsung_prism/models/logo_detector/best.pt`